# 金融行业研究报告助手：Qwen2.5-7B SFT → RM → DPO/PPO 训练闭环

> 目标：参考 MedicalGPT 的数据构造与训练范式，围绕金融垂直场景构建一个**可复用、可审计、可扩展**的端到端 Notebook。

本 Notebook 包含：
1. 数据读取与结构审查（`BAAI/IndustryInstruction_Finance-Economics`）；
2. 按 `docs/datasets.md` 的格式要求清洗与转换；
3. 面向“行业研究报告助手”的增强策略与偏好数据构建；
4. SFT、RM、DPO 与 PPO 的训练命令生成与执行模板。

## 0) 设计闭环（为什么这么做）

- **任务目标对齐**：行业研究报告助手更关注“结构化分析 + 证据 + 风险提示 + 合规表达”，而非泛闲聊。
- **数据闭环**：
  - SFT 负责“会答”；
  - RM 负责“好答”（可打分）；
  - DPO/PPO 负责“更偏好地答”（稳定朝业务目标优化）。
- **证据链**：每条数据尽可能保留 `source`、`difficulty`、`quality_score`、`safety_tag` 等元信息，支持回溯。

In [ ]:
# 如果本地环境缺少依赖，取消注释安装
# !pip install -U datasets pandas numpy tqdm scikit-learn

from __future__ import annotations
import json
import os
import re
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Any, Tuple

import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split

random.seed(42)
np.random.seed(42)

## 1) 读取数据集并审查结构

> 数据源：<https://huggingface.co/datasets/BAAI/IndustryInstruction_Finance-Economics>

如果当前环境无法访问 HuggingFace（例如 403/代理限制），可先手动下载到本地，再把 `DATASET_PATH` 指向本地 JSON/JSONL/Parquet。

In [ ]:
DATASET_NAME = "BAAI/IndustryInstruction_Finance-Economics"
DATASET_SPLIT = "train"
DATASET_PATH = None  # 例如: "./raw/finance_economics.jsonl"


def load_raw_dataset(dataset_name: str, split: str, dataset_path: str | None = None) -> Dataset:
    if dataset_path:
        ext = Path(dataset_path).suffix.lower()
        if ext in {".json", ".jsonl"}:
            ds = load_dataset("json", data_files=dataset_path, split="train")
        elif ext == ".parquet":
            ds = load_dataset("parquet", data_files=dataset_path, split="train")
        else:
            raise ValueError(f"Unsupported local file type: {ext}")
        return ds
    return load_dataset(dataset_name, split=split)


try:
    raw_ds = load_raw_dataset(DATASET_NAME, DATASET_SPLIT, DATASET_PATH)
    print(raw_ds)
    print("columns:", raw_ds.column_names)
    print("sample[0]:", raw_ds[0])
except Exception as e:
    print("[WARN] 加载失败，请检查网络/权限或改为本地文件。")
    print(type(e).__name__, str(e)[:400])
    raw_ds = None

## 2) 按 MedicalGPT 数据要求清洗与映射

`docs/datasets.md` 关键约束：
- **SFT**：`{"conversations": [{"from": "human", "value": ...}, {"from": "gpt", "value": ...}]}`
- **RM/DPO**：`{"question": ..., "response_chosen": ..., "response_rejected": ...}`
- **RL(PPO)**：`{"instruction": ..., "input": ..., "output": ...}`（SFT 可复用）

下面代码兼容常见字段命名（instruction/input/output、question/answer、prompt/response）。

In [ ]:
TEXT_KEYS_Q = ["instruction", "question", "query", "prompt", "input"]
TEXT_KEYS_A = ["output", "answer", "response", "completion", "target"]


def pick_first(d: Dict[str, Any], keys: List[str]) -> str:
    for k in keys:
        if k in d and d[k] is not None:
            v = str(d[k]).strip()
            if v:
                return v
    return ""


def normalize_record(rec: Dict[str, Any]) -> Dict[str, Any]:
    q = pick_first(rec, TEXT_KEYS_Q)
    a = pick_first(rec, TEXT_KEYS_A)

    # 若 question 在 input 且 instruction 更像系统前缀，则拼接
    ins = str(rec.get("instruction", "")).strip()
    inp = str(rec.get("input", "")).strip()
    if not q and ins:
        q = ins if not inp else f"{ins}

补充信息：{inp}"

    return {
        "question": q,
        "answer": a,
        "source": rec.get("source", "BAAI/IndustryInstruction_Finance-Economics"),
        "raw": rec,
    }


def quality_score(question: str, answer: str) -> float:
    score = 1.0
    # 过短惩罚
    if len(question) < 10:
        score -= 0.2
    if len(answer) < 30:
        score -= 0.4
    # 模板/拒答噪声惩罚
    noise_patterns = [r"作为AI", r"无法提供", r"我不能", r"抱歉", r"仅供参考"]
    hit = sum(bool(re.search(p, answer, flags=re.I)) for p in noise_patterns)
    score -= 0.08 * hit
    # 过度重复惩罚
    if len(set(answer.split())) < max(5, len(answer.split()) * 0.25):
        score -= 0.2
    return max(0.0, min(1.0, score))


def estimate_difficulty(question: str, answer: str) -> str:
    qlen, alen = len(question), len(answer)
    if qlen > 120 or alen > 600:
        return "hard"
    if qlen > 60 or alen > 240:
        return "medium"
    return "easy"


def clean_records(ds: Dataset, min_quality: float = 0.55) -> pd.DataFrame:
    rows = []
    for rec in ds:
        n = normalize_record(rec)
        q, a = n["question"], n["answer"]
        if not q or not a:
            continue
        s = quality_score(q, a)
        if s < min_quality:
            continue
        rows.append({
            "question": q,
            "answer": a,
            "quality_score": s,
            "difficulty": estimate_difficulty(q, a),
            "source": n["source"],
        })
    return pd.DataFrame(rows)


if raw_ds is not None:
    df = clean_records(raw_ds, min_quality=0.55)
    print(df.head(3).to_dict(orient="records"))
    print("clean size:", len(df))
    print("difficulty dist:
", df["difficulty"].value_counts(normalize=True).round(3))
else:
    df = pd.DataFrame(columns=["question", "answer", "quality_score", "difficulty", "source"])

## 3) 面向“行业研究报告助手”的数据增强

这里给出一个可落地的增强策略（可按你业务继续扩展）：

1. **公开数据**：监管披露、上市公司公告、研报摘要、宏观统计口径；
2. **自建数据**：内部行业模板、分析师历史问答、术语库；
3. **合成数据**：
   - 模板扩写（同一事实生成多种问法）；
   - 立场对照（乐观/中性/悲观）；
   - 报告结构化（结论、驱动因子、估值、风险、催化剂）。

> 注意：合成样本必须带 `synthetic=true` 与 `generator` 标记，后续可分桶观察是否污染风格。

In [ ]:
REPORT_FRAME = """请以行业研究报告风格回答，并包含：
1) 核心结论（3点内）
2) 关键驱动因子
3) 估值/比较视角
4) 风险与反例
5) 后续跟踪指标
""".strip()


def augment_for_research_assistant(df: pd.DataFrame, n_per_sample: int = 1) -> pd.DataFrame:
    rows = []
    for _, r in df.iterrows():
        rows.append({**r.to_dict(), "synthetic": False, "generator": "raw"})
        for _ in range(n_per_sample):
            q2 = f"{REPORT_FRAME}

问题：{r['question']}"
            a2 = r["answer"]
            rows.append({
                "question": q2,
                "answer": a2,
                "quality_score": min(1.0, float(r["quality_score"]) + 0.03),
                "difficulty": r["difficulty"],
                "source": r["source"],
                "synthetic": True,
                "generator": "template_rewrite_v1",
            })
    out = pd.DataFrame(rows)
    return out.sample(frac=1.0, random_state=42).reset_index(drop=True)


aug_df = augment_for_research_assistant(df, n_per_sample=1) if len(df) else df.copy()
print("aug size:", len(aug_df))
if len(aug_df):
    print(aug_df[["synthetic", "generator"]].value_counts())

## 4) 构造偏好数据（DPO/RM）

### 偏好数据来源建议
- **人工标注**（优先）：金融从业者/研究员按 rubric 打分；
- **半自动合成**：
  - `chosen`：结构完整、风险披露充分、避免过度确定性语言；
  - `rejected`：信息遗漏、逻辑跳步、夸大收益、无风险提示；
- **控噪**：
  - 交叉复核（至少双标）；
  - 一致性阈值过滤（如 Cohen's κ）；
  - 设 hard-negative 样本比例上限，防止模型学偏。

In [ ]:
RISK_HINT = "风险提示：以上分析不构成投资建议，需结合最新披露与市场流动性变化。"


def make_preference_pair(question: str, answer: str) -> Tuple[str, str]:
    chosen = answer.strip()
    if RISK_HINT not in chosen:
        chosen = chosen + "

" + RISK_HINT

    rejected = re.sub(r"风险|不确定|波动|假设", "", answer)
    rejected = rejected[: max(30, int(len(rejected) * 0.7))]
    if len(rejected) < 30:
        rejected = "结论看多，预计显著上涨，细节略。"
    return chosen, rejected


def build_preference_df(df: pd.DataFrame, max_samples: int = 50000) -> pd.DataFrame:
    rows = []
    for _, r in df.head(max_samples).iterrows():
        c, rej = make_preference_pair(r["question"], r["answer"])
        rows.append({
            "question": r["question"],
            "response_chosen": c,
            "response_rejected": rej,
            "source": r.get("source", "unknown"),
            "difficulty": r.get("difficulty", "medium"),
        })
    return pd.DataFrame(rows)


pref_df = build_preference_df(aug_df) if len(aug_df) else pd.DataFrame(
    columns=["question", "response_chosen", "response_rejected", "source", "difficulty"]
)
print("preference size:", len(pref_df))
print(pref_df.head(2).to_dict(orient="records") if len(pref_df) else "empty")

## 5) 导出为 MedicalGPT 可直接训练的格式

In [ ]:
OUT_DIR = Path("data/finance")
OUT_DIR.mkdir(parents=True, exist_ok=True)


def to_sft_jsonl(df: pd.DataFrame, path: Path):
    with path.open("w", encoding="utf-8") as f:
        for _, r in df.iterrows():
            obj = {
                "conversations": [
                    {"from": "human", "value": r["question"]},
                    {"from": "gpt", "value": r["answer"]},
                ]
            }
            f.write(json.dumps(obj, ensure_ascii=False) + "
")


def to_rm_jsonl(df: pd.DataFrame, path: Path):
    with path.open("w", encoding="utf-8") as f:
        for _, r in df.iterrows():
            obj = {
                "question": r["question"],
                "response_chosen": r["response_chosen"],
                "response_rejected": r["response_rejected"],
            }
            f.write(json.dumps(obj, ensure_ascii=False) + "
")


def to_rl_jsonl(df: pd.DataFrame, path: Path):
    with path.open("w", encoding="utf-8") as f:
        for _, r in df.iterrows():
            obj = {
                "instruction": r["question"],
                "input": "",
                "output": r["answer"],
            }
            f.write(json.dumps(obj, ensure_ascii=False) + "
")


def split_df(df: pd.DataFrame, test_size=0.02, val_size=0.03):
    if len(df) < 100:
        return df, df.iloc[:0].copy(), df.iloc[:0].copy()
    train, test = train_test_split(df, test_size=test_size, random_state=42)
    train, val = train_test_split(train, test_size=val_size, random_state=42)
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)


sft_train, sft_val, sft_test = split_df(aug_df)
rm_train, rm_val, rm_test = split_df(pref_df)

sft_train_path = OUT_DIR / "finance_sft_train.jsonl"
sft_val_path = OUT_DIR / "finance_sft_val.jsonl"
rm_train_path = OUT_DIR / "finance_rm_train.jsonl"
rm_val_path = OUT_DIR / "finance_rm_val.jsonl"
rl_train_path = OUT_DIR / "finance_rl_train.jsonl"

to_sft_jsonl(sft_train, sft_train_path)
to_sft_jsonl(sft_val, sft_val_path)
to_rm_jsonl(rm_train, rm_train_path)
to_rm_jsonl(rm_val, rm_val_path)
to_rl_jsonl(sft_train, rl_train_path)

print("saved:", sft_train_path, sft_val_path, rm_train_path, rm_val_path, rl_train_path)
print("sizes:", len(sft_train), len(sft_val), len(rm_train), len(rm_val))

## 6) 训练：SFT → RM → DPO/PPO

> 基座模型：`Qwen2.5-7B`

根据你机器资源调整 batch、gradient_accumulation、LoRA、量化参数。

In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-7B"

SFT_CMD = f"""
python supervised_finetuning.py \
  --model_name_or_path {BASE_MODEL} \
  --train_file_dir data/finance \
  --validation_file_dir data/finance \
  --per_device_train_batch_size 1 \
  --per_device_eval_batch_size 1 \
  --do_train --do_eval \
  --use_peft True --lora_rank 8 --lora_alpha 16 --lora_dropout 0.05 \
  --max_train_samples 200000 \
  --num_train_epochs 2 \
  --learning_rate 2e-5 \
  --output_dir outputs-finance-sft
""".strip()

RM_CMD = f"""
python reward_modeling.py \
  --model_name_or_path outputs-finance-sft \
  --train_file_dir data/finance \
  --validation_file_dir data/finance \
  --per_device_train_batch_size 1 \
  --per_device_eval_batch_size 1 \
  --do_train --do_eval \
  --num_train_epochs 1 \
  --learning_rate 1e-5 \
  --output_dir outputs-finance-rm
""".strip()

DPO_CMD = f"""
python dpo_training.py \
  --model_name_or_path outputs-finance-sft \
  --train_file_dir data/finance \
  --validation_file_dir data/finance \
  --beta 0.1 \
  --per_device_train_batch_size 1 \
  --gradient_accumulation_steps 8 \
  --num_train_epochs 1 \
  --learning_rate 1e-6 \
  --output_dir outputs-finance-dpo
""".strip()

PPO_CMD = f"""
python ppo_training.py \
  --sft_model_name_or_path outputs-finance-sft \
  --reward_model_name_or_path outputs-finance-rm \
  --train_file_dir data/finance \
  --output_dir outputs-finance-ppo \
  --per_device_train_batch_size 1 \
  --mini_batch_size 1 \
  --ppo_epochs 1
""".strip()

print("===== SFT =====
", SFT_CMD)
print("
===== RM =====
", RM_CMD)
print("
===== DPO =====
", DPO_CMD)
print("
===== PPO =====
", PPO_CMD)

## 7) 可选：直接在 Notebook 中执行（按需取消注释）

In [ ]:
# !{SFT_CMD}
# !{RM_CMD}
# !{DPO_CMD}
# !{PPO_CMD}

## 8) 评测集构建建议（离线 + 在线）

- **离线评测集**（建议分层抽样）：
  - 行业覆盖：金融、地产、医药、消费、制造、科技；
  - 能力覆盖：摘要、归因、对比、预测、风险披露、估值；
  - 难度覆盖：基础事实 / 多跳推理 / 长上下文证据整合。
- **指标**：
  - 通用：Win-rate、长度归一后的偏好分；
  - 业务：结构完整性、证据引用准确性、风险披露充分性、术语规范性。
- **在线评估**：
  - 分析师 A/B 打分 + 用户点击/停留 + 下游投研任务完成率。

In [ ]:
# 快速 sanity check：导出文件是否符合 MedicalGPT 需要的最小字段

def check_jsonl(path: Path, required_keys: List[str], n: int = 3):
    ok = True
    with path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            obj = json.loads(line)
            for k in required_keys:
                if k not in obj:
                    ok = False
            if i < n:
                print(path.name, "sample", i, obj)
    print(path.name, "PASS" if ok else "FAIL")

if sft_train_path.exists():
    check_jsonl(sft_train_path, ["conversations"])
if rm_train_path.exists():
    check_jsonl(rm_train_path, ["question", "response_chosen", "response_rejected"])
if rl_train_path.exists():
    check_jsonl(rl_train_path, ["instruction", "input", "output"])